# 04 — Tabular feature EDA

**Child Mind Institute — Problematic Internet Use**

Этот ноутбук объединяет две задачи:

1. **Feature distributions** — какие допустимые tabular features доступны, как они распределены, насколько они skewed, где есть подозрительные значения и как соотносятся train/test.
2. **Feature–target relationships** — как эти признаки связаны с `sii`, если исключить `PCIAT-*`, `id` и сам target.

Таким образом, ноутбук заменяет два отдельных файла:

- `04_feature_distributions.ipynb`
- `05_feature_target_relationships.ipynb`

> Actigraphy/parquet данные здесь не используются.


## 1. Imports and data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, pearsonr, rankdata, spearmanr
from sklearn.feature_selection import mutual_info_classif

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

TARGET = "sii"
ID_COL = "id"
AGE_COL = "Basic_Demos-Age"
RANDOM_STATE = 42


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
DATA_DICT_PATH = DATA_DIR / "data_dictionary.csv"

for path in [TRAIN_PATH, TEST_PATH, DATA_DICT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
data_dict = pd.read_csv(DATA_DICT_PATH)

labeled = train.loc[train[TARGET].notna()].copy()
MODEL_FEATURES = [c for c in test.columns if c != ID_COL]

feature_meta = (
    data_dict.loc[data_dict["Field"].isin(MODEL_FEATURES)]
    .set_index("Field")
    .reindex(MODEL_FEATURES)
)

NUMERIC_MEASURE_COLS = [
    c for c in MODEL_FEATURES
    if feature_meta.loc[c, "Type"] in {"float", "int"}
]

CATEGORICAL_INT_COLS = [
    c for c in MODEL_FEATURES
    if feature_meta.loc[c, "Type"] == "categorical int"
]

STRING_COLS = [
    c for c in MODEL_FEATURES
    if feature_meta.loc[c, "Type"] == "str"
]

CATEGORICAL_COLS = CATEGORICAL_INT_COLS + STRING_COLS

print("Train:", train.shape)
print("Test:", test.shape)
print("Labeled rows:", labeled.shape)
print("Model features:", len(MODEL_FEATURES))
print("Numeric measurements:", len(NUMERIC_MEASURE_COLS))
print("Categorical int:", len(CATEGORICAL_INT_COLS))
print("String categories:", len(STRING_COLS))


## 2. Feature taxonomy

Не разделяем признаки только по pandas `dtype`. Используем `data_dictionary.csv`, потому что часть колонок имеет тип `categorical int` и по смыслу должна анализироваться как категориальная.


In [ ]:
type_counts = (
    feature_meta["Type"]
    .value_counts(dropna=False)
    .rename("n_features")
    .to_frame()
)

display(type_counts)


## 3. Numeric feature summary

In [ ]:
numeric_rows = []

for col in NUMERIC_MEASURE_COLS:
    s = train[col]
    x = s.dropna()

    numeric_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "description": feature_meta.loc[col, "Description"],
        "n_available": len(x),
        "missing_%": s.isna().mean() * 100,
        "n_unique": x.nunique(),
        "mean": x.mean(),
        "std": x.std(),
        "min": x.min(),
        "p01": x.quantile(0.01),
        "p25": x.quantile(0.25),
        "median": x.median(),
        "p75": x.quantile(0.75),
        "p99": x.quantile(0.99),
        "max": x.max(),
        "skew": x.skew(),
    })

numeric_summary = (
    pd.DataFrame(numeric_rows)
    .sort_values(["instrument", "feature"])
    .reset_index(drop=True)
)

display(numeric_summary.round(3))


### Most skewed numeric features

In [ ]:
skew_summary = (
    numeric_summary[
        ["feature", "instrument", "n_available", "missing_%", "skew"]
    ]
    .assign(abs_skew=lambda x: x["skew"].abs())
    .sort_values("abs_skew", ascending=False)
)

display(skew_summary.head(20).round(3))


In [ ]:
top_skew = skew_summary.dropna(subset=["skew"]).head(15).sort_values("skew")

plt.figure(figsize=(10, 6))
plt.barh(top_skew["feature"], top_skew["skew"])
plt.axvline(0)
plt.xlabel("Skewness")
plt.ylabel("")
plt.title("Most skewed numeric model features")
plt.tight_layout()
plt.show()


### Representative numeric distributions

In [ ]:
PLOT_FEATURES = [
    "Basic_Demos-Age",
    "CGAS-CGAS_Score",
    "Physical-BMI",
    "Physical-Height",
    "Physical-Weight",
    "Fitness_Endurance-Max_Stage",
    "PAQ_C-PAQ_C_Total",
    "PAQ_A-PAQ_A_Total",
    "SDS-SDS_Total_Raw",
    "BIA-BIA_Fat",
]

PLOT_FEATURES = [
    c for c in PLOT_FEATURES
    if c in NUMERIC_MEASURE_COLS
]

def plot_numeric_distribution(df, feature, clip=(0.005, 0.995), bins=30):
    s = df[feature].dropna()

    if len(s) == 0:
        return

    plot_s = s

    if clip is not None and s.nunique() > 2:
        low, high = s.quantile(list(clip))
        plot_s = s[s.between(low, high)]

    plt.figure(figsize=(8, 4))
    plt.hist(plot_s, bins=bins, edgecolor="black")
    plt.xlabel(feature)
    plt.ylabel("Participants")
    plt.title(
        f"{feature}\n"
        f"n={len(s)}, missing={df[feature].isna().mean():.1%}"
    )
    plt.tight_layout()
    plt.show()

for feature in PLOT_FEATURES:
    plot_numeric_distribution(train, feature)


## 4. Categorical feature distributions

In [ ]:
categorical_int_summary = []

for col in CATEGORICAL_INT_COLS:
    values = sorted(train[col].dropna().unique().tolist())

    categorical_int_summary.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "description": feature_meta.loc[col, "Description"],
        "dictionary_values": feature_meta.loc[col, "Values"],
        "value_labels": feature_meta.loc[col, "Value Labels"],
        "observed_values": values,
        "n_unique": len(values),
        "missing_%": train[col].isna().mean() * 100,
    })

categorical_int_summary = pd.DataFrame(categorical_int_summary)

display(categorical_int_summary)


In [ ]:
string_summary = []

for col in STRING_COLS:
    observed = sorted(train[col].dropna().astype(str).unique())

    string_summary.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "description": feature_meta.loc[col, "Description"],
        "dictionary_values": feature_meta.loc[col, "Values"],
        "observed_values": observed,
        "n_unique": len(observed),
        "missing_%": train[col].isna().mean() * 100,
    })

string_summary = pd.DataFrame(string_summary)

display(string_summary)


In [ ]:
season_distribution = pd.DataFrame({
    col: (
        train[col]
        .value_counts(normalize=True)
        .mul(100)
    )
    for col in STRING_COLS
}).T

display(season_distribution.round(1))


## 5. IQR-based outlier screening

In [ ]:
outlier_rows = []

for col in NUMERIC_MEASURE_COLS:
    s = train[col].dropna()

    if len(s) < 4:
        continue

    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1

    if iqr == 0:
        low = q1
        high = q3
        n_out = 0
    else:
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        n_out = ((s < low) | (s > high)).sum()

    outlier_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "n_available": len(s),
        "iqr_low": low,
        "iqr_high": high,
        "n_iqr_outliers": n_out,
        "outlier_%": 100 * n_out / len(s),
        "min": s.min(),
        "p01": s.quantile(0.01),
        "median": s.median(),
        "p99": s.quantile(0.99),
        "max": s.max(),
    })

outlier_summary = (
    pd.DataFrame(outlier_rows)
    .sort_values("outlier_%", ascending=False)
    .reset_index(drop=True)
)

display(outlier_summary.head(25).round(3))


In [ ]:
top_outliers = outlier_summary.head(15).sort_values("outlier_%")

plt.figure(figsize=(10, 6))
plt.barh(top_outliers["feature"], top_outliers["outlier_%"])
plt.xlabel("IQR outliers, % of available observations")
plt.ylabel("")
plt.title("Features with the highest IQR-outlier fraction")
plt.tight_layout()
plt.show()


### Extreme-value screening

In [ ]:
extreme_summary = numeric_summary[
    [
        "feature",
        "instrument",
        "min",
        "p01",
        "median",
        "p99",
        "max",
    ]
].copy()

eps = 1e-12

extreme_summary["upper_gap_scaled"] = (
    (extreme_summary["max"] - extreme_summary["p99"]).abs()
    / ((extreme_summary["p99"] - extreme_summary["median"]).abs() + eps)
)

extreme_summary["lower_gap_scaled"] = (
    (extreme_summary["p01"] - extreme_summary["min"]).abs()
    / ((extreme_summary["median"] - extreme_summary["p01"]).abs() + eps)
)

extreme_summary["max_extreme_score"] = extreme_summary[
    ["upper_gap_scaled", "lower_gap_scaled"]
].max(axis=1)

extreme_summary = extreme_summary.sort_values(
    "max_extreme_score",
    ascending=False,
)

display(extreme_summary.head(20).round(3))


In [ ]:
EXTREME_FEATURES_TO_INSPECT = (
    extreme_summary["feature"]
    .head(8)
    .tolist()
)

for col in EXTREME_FEATURES_TO_INSPECT:
    print(f"\n{col}")

    view = (
        train[[ID_COL, col]]
        .dropna()
        .sort_values(col)
    )

    display(
        pd.concat([view.head(3), view.tail(3)]).drop_duplicates()
    )


## 6. Train/test compatibility

In [ ]:
unseen_category_rows = []

for col in CATEGORICAL_COLS:
    train_values = set(train[col].dropna().astype(str))
    test_values = set(test[col].dropna().astype(str))

    unseen_category_rows.append({
        "feature": col,
        "train_categories": sorted(train_values),
        "test_categories": sorted(test_values),
        "unseen_in_train": sorted(test_values - train_values),
    })

unseen_categories = pd.DataFrame(unseen_category_rows)

display(unseen_categories)


In [ ]:
range_rows = []

for col in NUMERIC_MEASURE_COLS:
    train_s = train[col].dropna()
    test_s = test[col].dropna()

    if len(train_s) == 0 or len(test_s) == 0:
        continue

    outside = ((test_s < train_s.min()) | (test_s > train_s.max()))

    range_rows.append({
        "feature": col,
        "train_min": train_s.min(),
        "train_max": train_s.max(),
        "test_min": test_s.min(),
        "test_max": test_s.max(),
        "n_test_available": len(test_s),
        "n_test_outside_train_range": outside.sum(),
    })

range_check = pd.DataFrame(range_rows)

display(
    range_check.sort_values(
        "n_test_outside_train_range",
        ascending=False,
    ).round(3)
)


In [ ]:
median_compare = []

for col in NUMERIC_MEASURE_COLS:
    train_s = train[col].dropna()
    test_s = test[col].dropna()

    if len(train_s) == 0 or len(test_s) == 0:
        continue

    iqr = train_s.quantile(0.75) - train_s.quantile(0.25)

    median_compare.append({
        "feature": col,
        "train_median": train_s.median(),
        "test_median": test_s.median(),
        "train_iqr": iqr,
        "median_difference": test_s.median() - train_s.median(),
        "median_diff_in_train_iqr": (
            (test_s.median() - train_s.median()) / iqr
            if iqr != 0 else np.nan
        ),
        "n_test_available": len(test_s),
    })

median_compare = (
    pd.DataFrame(median_compare)
    .assign(abs_scaled_diff=lambda x: x["median_diff_in_train_iqr"].abs())
    .sort_values("abs_scaled_diff", ascending=False)
)

display(median_compare.head(20).round(3))


## 7. Spearman correlation with SII

Так как `sii` — ordinal target, для первичного screening continuous features используем Spearman correlation.


In [ ]:
spearman_rows = []

for col in NUMERIC_MEASURE_COLS:
    sub = labeled[[col, TARGET]].dropna()

    if len(sub) < 10 or sub[col].nunique() < 2:
        continue

    rho, p_value = spearmanr(sub[col], sub[TARGET])

    spearman_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "n": len(sub),
        "rho": rho,
        "p_value": p_value,
        "abs_rho": abs(rho),
    })

spearman_summary = (
    pd.DataFrame(spearman_rows)
    .sort_values("abs_rho", ascending=False)
    .reset_index(drop=True)
)

display(spearman_summary.head(25).round(4))


In [ ]:
top_spearman = spearman_summary.head(15).sort_values("rho")

plt.figure(figsize=(10, 6))
plt.barh(top_spearman["feature"], top_spearman["rho"])
plt.axvline(0)
plt.xlabel("Spearman rho with SII")
plt.ylabel("")
plt.title("Strongest raw ordinal associations")
plt.tight_layout()
plt.show()


### Age confounding

Часть сильных raw correlations для physical/BIA features может быть объяснена тем, что и сами признаки, и `sii` связаны с возрастом. Поэтому next step — частичная корреляция с контролем `Basic_Demos-Age`.


## 8. Partial Spearman controlling for age

In [ ]:
def partial_spearman_controlling_age(df, feature, target=TARGET, age=AGE_COL):
    sub = df[[feature, target, age]].dropna()

    if (
        len(sub) < 10
        or sub[feature].nunique() < 2
        or sub[age].nunique() < 2
    ):
        return np.nan, np.nan, len(sub)

    x_rank = rankdata(sub[feature].to_numpy())
    y_rank = rankdata(sub[target].to_numpy())
    age_rank = rankdata(sub[age].to_numpy())

    X = np.column_stack([np.ones(len(sub)), age_rank])

    beta_x = np.linalg.lstsq(X, x_rank, rcond=None)[0]
    beta_y = np.linalg.lstsq(X, y_rank, rcond=None)[0]

    residual_x = x_rank - X @ beta_x
    residual_y = y_rank - X @ beta_y

    r, p_value = pearsonr(residual_x, residual_y)

    return r, p_value, len(sub)


In [ ]:
partial_rows = []

for col in NUMERIC_MEASURE_COLS:
    if col == AGE_COL:
        continue

    rho, p_value, n = partial_spearman_controlling_age(labeled, col)

    if np.isnan(rho):
        continue

    raw_match = spearman_summary.loc[
        spearman_summary["feature"].eq(col),
        "rho",
    ]

    raw_rho = raw_match.iloc[0] if len(raw_match) else np.nan

    partial_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "n": n,
        "raw_rho": raw_rho,
        "partial_rho_age": rho,
        "p_value": p_value,
        "abs_partial_rho": abs(rho),
        "delta_abs_rho": abs(raw_rho) - abs(rho),
    })

partial_spearman = (
    pd.DataFrame(partial_rows)
    .sort_values("abs_partial_rho", ascending=False)
    .reset_index(drop=True)
)

display(partial_spearman.head(25).round(4))


In [ ]:
top_partial = partial_spearman.head(15).sort_values("partial_rho_age")

plt.figure(figsize=(10, 6))
plt.barh(top_partial["feature"], top_partial["partial_rho_age"])
plt.axvline(0)
plt.xlabel("Partial Spearman rho with SII")
plt.ylabel("")
plt.title("Associations after controlling for age")
plt.tight_layout()
plt.show()


In [ ]:
comparison = (
    partial_spearman
    .sort_values("delta_abs_rho", ascending=False)
    .head(20)
)

display(
    comparison[
        ["feature", "raw_rho", "partial_rho_age", "delta_abs_rho", "n"]
    ].round(4)
)


## 9. Categorical features vs SII

In [ ]:
def cramers_v(x, y):
    table = pd.crosstab(x, y)

    if min(table.shape) < 2:
        return np.nan

    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()

    if n == 0:
        return np.nan

    phi2 = chi2 / n
    r, k = table.shape

    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    r_corr = r - ((r - 1) ** 2) / max(n - 1, 1)
    k_corr = k - ((k - 1) ** 2) / max(n - 1, 1)

    denom = min(k_corr - 1, r_corr - 1)

    if denom <= 0:
        return np.nan

    return np.sqrt(phi2_corr / denom)

categorical_rows = []

for col in CATEGORICAL_COLS:
    x = (
        labeled[col]
        .astype("object")
        .where(labeled[col].notna(), "__MISSING__")
        .astype(str)
    )

    v = cramers_v(x, labeled[TARGET].astype(int))

    categorical_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "n_categories": x.nunique(),
        "missing_%": labeled[col].isna().mean() * 100,
        "cramers_v": v,
    })

categorical_association = (
    pd.DataFrame(categorical_rows)
    .sort_values("cramers_v", ascending=False)
    .reset_index(drop=True)
)

display(categorical_association.round(4))


In [ ]:
plot_cat = (
    categorical_association
    .dropna(subset=["cramers_v"])
    .head(15)
    .sort_values("cramers_v")
)

plt.figure(figsize=(10, 6))
plt.barh(plot_cat["feature"], plot_cat["cramers_v"])
plt.xlabel("Cramér's V with SII")
plt.ylabel("")
plt.title("Categorical associations with SII")
plt.tight_layout()
plt.show()


## 10. Univariate mutual information

In [ ]:
mi_rows = []

y = labeled[TARGET].astype(int).to_numpy()

for col in MODEL_FEATURES:
    feature_type = feature_meta.loc[col, "Type"]

    if feature_type in {"float", "int"}:
        s = labeled[col].astype(float)

        if s.notna().sum() == 0:
            continue

        x = s.fillna(s.median()).to_numpy().reshape(-1, 1)
        discrete = False
    else:
        s = (
            labeled[col]
            .astype("object")
            .where(labeled[col].notna(), "__MISSING__")
            .astype(str)
        )

        codes, _ = pd.factorize(s)
        x = codes.reshape(-1, 1)
        discrete = True

    if np.unique(x).size < 2:
        continue

    mi = mutual_info_classif(
        x,
        y,
        discrete_features=discrete,
        random_state=RANDOM_STATE,
    )[0]

    mi_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        "type": feature_type,
        "mutual_information": mi,
        "missing_%": labeled[col].isna().mean() * 100,
    })

mi_summary = (
    pd.DataFrame(mi_rows)
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(mi_summary.head(25).round(4))


In [ ]:
top_mi = mi_summary.head(15).sort_values("mutual_information")

plt.figure(figsize=(10, 6))
plt.barh(top_mi["feature"], top_mi["mutual_information"])
plt.xlabel("Mutual information with SII")
plt.ylabel("")
plt.title("Top univariate MI features")
plt.tight_layout()
plt.show()


## 11. Missingness itself vs SII

In [ ]:
missing_by_class_rows = []
classes = sorted(labeled[TARGET].astype(int).unique())

for col in MODEL_FEATURES:
    rates = {}

    for cls in classes:
        cls_mask = labeled[TARGET].eq(cls)
        rates[cls] = labeled.loc[cls_mask, col].isna().mean() * 100

    values = list(rates.values())

    missing_by_class_rows.append({
        "feature": col,
        "instrument": feature_meta.loc[col, "Instrument"],
        **{f"missing_sii_{cls}_%": rates[cls] for cls in classes},
        "max_missing_gap_pp": max(values) - min(values),
    })

missingness_target = (
    pd.DataFrame(missing_by_class_rows)
    .sort_values("max_missing_gap_pp", ascending=False)
    .reset_index(drop=True)
)

display(missingness_target.head(25).round(2))


## 12. Class-wise distributions of selected features

In [ ]:
PLOT_FEATURES_TARGET = [
    "SDS-SDS_Total_Raw",
    "SDS-SDS_Total_T",
    "CGAS-CGAS_Score",
    "Physical-BMI",
    "Physical-HeartRate",
    "Fitness_Endurance-Max_Stage",
]

PLOT_FEATURES_TARGET = [
    col for col in PLOT_FEATURES_TARGET
    if col in labeled.columns
]

def plot_feature_by_sii(df, feature):
    groups = []
    labels = []

    for cls in sorted(df[TARGET].dropna().astype(int).unique()):
        values = df.loc[df[TARGET].eq(cls), feature].dropna()

        if len(values) == 0:
            continue

        groups.append(values)
        labels.append(f"{cls}\n(n={len(values)})")

    if not groups:
        return

    plt.figure(figsize=(8, 4))
    plt.boxplot(groups, labels=labels, showfliers=False)
    plt.xlabel("SII")
    plt.ylabel(feature)
    plt.title(f"{feature} by SII")
    plt.tight_layout()
    plt.show()

for feature in PLOT_FEATURES_TARGET:
    plot_feature_by_sii(labeled, feature)


In [ ]:
summary_features = list(dict.fromkeys(
    partial_spearman["feature"].head(10).tolist()
    + ["Basic_Demos-Age", "CGAS-CGAS_Score", "Physical-BMI"]
))

summary_features = [c for c in summary_features if c in labeled.columns]

class_summary_rows = []

for col in summary_features:
    for cls in sorted(labeled[TARGET].astype(int).unique()):
        s = labeled.loc[labeled[TARGET].eq(cls), col].dropna()

        if len(s) == 0:
            continue

        class_summary_rows.append({
            "feature": col,
            "sii": cls,
            "n": len(s),
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(),
        })

class_numeric_summary = pd.DataFrame(class_summary_rows)

display(class_numeric_summary.set_index(["feature", "sii"]).round(3))


## 13. Consolidated screening table

In [ ]:
ranking = (
    mi_summary[
        ["feature", "instrument", "type", "mutual_information", "missing_%"]
    ]
    .copy()
)

ranking = ranking.merge(
    spearman_summary[["feature", "rho", "abs_rho"]],
    on="feature",
    how="left",
)

ranking = ranking.merge(
    partial_spearman[
        ["feature", "partial_rho_age", "abs_partial_rho"]
    ],
    on="feature",
    how="left",
)

ranking = ranking.merge(
    categorical_association[["feature", "cramers_v"]],
    on="feature",
    how="left",
)

ranking = ranking.merge(
    missingness_target[["feature", "max_missing_gap_pp"]],
    on="feature",
    how="left",
)

display(
    ranking
    .sort_values("mutual_information", ascending=False)
    .head(30)
    .round(4)
)


## 14. Main findings to carry forward

1. Среди 58 допустимых model features доступны как continuous measurements, так и categorical variables; `categorical int` не должны обрабатываться как обычные continuous numbers.
2. Numeric features сильно различаются по масштабу, missingness и skewness; особенно выделяются признаки из блоков `BIA`, `Physical` и `Fitness`.
3. IQR-screening и extreme-value screening показывают наличие явно подозрительных значений, особенно в ряде `BIA`-признаков и в `CGAS-CGAS_Score`; их стоит проверить отдельно до обучения моделей.
4. В supplied `test.csv` не обнаружено unseen categories и значений, выходящих за полный observed train range, но из-за очень малого размера test эти проверки носят только sanity-check характер.
5. Raw Spearman ranking поднимает возраст, size-related physical measures и ряд BIA features, однако значительная часть этих связей ослабевает после контроля возраста.
6. Возраст является сильным confounder для интерпретации feature–target relationships в этом dataset.
7. После age adjustment среди числовых признаков особенно устойчиво выглядят `SDS-SDS_Total_Raw` и `SDS-SDS_Total_T`, что делает блок Sleep Disturbance Scale одним из наиболее интересных для дальнейшего анализа.
8. Mutual information полезен как дополнительный univariate ranking heuristic, но не заменяет полноценную validation-based feature importance.
9. Missingness сам по себе различается между классами `sii` и потенциально может нести predictive information, однако часть этого эффекта структурно связана с возрастом и полнотой обследования.

